![Kayak](https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png)

# Plan Your Trip with Kayak — Solution Complete

**Auteur** : Jedha Bootcamp — Data Engineering  
**Date** : Fevrier 2026  
**Stack** : Python, Nominatim, OpenWeatherMap, Selenium, BeautifulSoup, AWS S3, AWS RDS PostgreSQL, Plotly

---

## Architecture du projet

```
APIs (Nominatim + OpenWeatherMap)
     |
Web Scraping (Booking.com via Selenium)
     |
Traitement & Scoring (Pandas)
     |
AWS S3 (Data Lake -- donnees brutes)
     |
ETL (Extract -> Transform -> Load)
     |
AWS RDS PostgreSQL (Data Warehouse)
     |
Visualisation Plotly (Cartes interactives)
```

| Etape | Description |
|-------|-------------|
| **0** | Installation & Configuration |
| **1** | Coordonnees GPS via Nominatim |
| **2** | Donnees meteo via OpenWeatherMap |
| **3** | Score meteo & classement |
| **4** | Scraping hotels Booking.com |
| **5** | Upload vers S3 (Data Lake) |
| **6** | ETL vers AWS RDS (Data Warehouse) |
| **7** | Visualisation Plotly |

---
# ETAPE 0 — Installation & Configuration

In [ ]:
# Installation des librairies (executer une seule fois)
!pip install requests pandas plotly boto3 sqlalchemy psycopg2-binary selenium \
             beautifulsoup4 python-dotenv tqdm webdriver-manager -q

In [ ]:
# Imports
import os
import re
import io
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

# Web Scraping
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Cloud & BDD
import boto3
from sqlalchemy import create_engine, text

# Visualisation
import plotly.express as px
import plotly.graph_objects as go

print('Imports OK')

In [ ]:
# Chargement des variables depuis .env
load_dotenv()

OWM_API_KEY           = os.getenv('OWM_API_KEY')
AWS_ACCESS_KEY_ID     = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = os.getenv('AWS_DEFAULT_REGION', 'eu-west-3')
S3_BUCKET             = os.getenv('S3_BUCKET')
RDS_URI               = os.getenv('RDS_URI')

# 35 villes françaises cibles
CITIES = [
    'Mont Saint Michel', 'St Malo', 'Bayeux', 'Le Havre', 'Rouen',
    'Paris', 'Amiens', 'Lille', 'Strasbourg', 'Chateau du Haut Koenigsbourg',
    'Colmar', 'Eguisheim', 'Besancon', 'Dijon', 'Annecy',
    'Grenoble', 'Lyon', 'Gorges du Verdon', 'Bormes les Mimosas', 'Cassis',
    'Marseille', 'Aix en Provence', 'Avignon', 'Uzes', 'Nimes',
    'Aigues Mortes', 'Saintes Maries de la mer', 'Collioure', 'Carcassonne', 'Ariege',
    'Toulouse', 'Montauban', 'Biarritz', 'Bayonne', 'La Rochelle'
]

print(f'{len(CITIES)} villes chargees')
print(f'OWM : {"OK" if OWM_API_KEY else "MANQUANT"}')
print(f'AWS S3 : {"OK" if AWS_ACCESS_KEY_ID else "MANQUANT"}')
print(f'RDS : {"OK" if RDS_URI and "ENDPOINT" not in RDS_URI else "MANQUANT"}')

---
# ETAPE 1 — Coordonnees GPS avec Nominatim

L'API **Nominatim** (OpenStreetMap) convertit un nom de ville en coordonnees GPS.  
Gratuite, sans cle API, avec un delai de 1s entre requetes (politique d'utilisation).

In [ ]:
def get_coordinates(city_name):
    """Recupere lat/lon d'une ville via Nominatim."""
    url = 'https://nominatim.openstreetmap.org/search'
    params = {'q': f'{city_name}, France', 'format': 'json', 'limit': 1}
    headers = {'User-Agent': 'KayakProjectJedha/1.0'}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        if data:
            return {'city': city_name, 'lat': float(data[0]['lat']), 'lon': float(data[0]['lon'])}
    except Exception as e:
        print(f'Erreur {city_name}: {e}')
    return None

In [ ]:
coords_list = []
for city in tqdm(CITIES, desc='GPS'):
    result = get_coordinates(city)
    if result:
        coords_list.append(result)
    time.sleep(1)

df_coords = pd.DataFrame(coords_list)
df_coords['city_id'] = range(1, len(df_coords) + 1)

print(f'{len(df_coords)} villes geolocalises')
df_coords.head()

---
# ETAPE 2 — Donnees Meteo via OpenWeatherMap

On utilise l'**API Forecast** gratuite d'OWM (5 jours, intervalles de 3h).  
On agregge les donnees par jour pour obtenir : temp max, proba pluie, volume pluie, humidite.

> Note : L'API One Call 3.0 necessite un abonnement payant. L'API Forecast est gratuite et suffisante pour le projet.

In [ ]:
def get_weather_forecast(lat, lon, city_id, city_name):
    """Recupere les previsions meteo 5 jours (API Forecast OWM)."""
    url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {
        'lat': lat, 'lon': lon, 'appid': OWM_API_KEY,
        'units': 'metric', 'lang': 'fr', 'cnt': 40
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if r.status_code != 200:
            return []

        daily = {}
        for item in data.get('list', []):
            date = pd.to_datetime(item['dt'], unit='s').strftime('%Y-%m-%d')
            if date not in daily:
                daily[date] = {'temps': [], 'humidity': [], 'pop': [], 'rain_mm': 0, 'wind_speeds': []}
            daily[date]['temps'].append(item['main']['temp'])
            daily[date]['humidity'].append(item['main']['humidity'])
            daily[date]['pop'].append(item.get('pop', 0))
            daily[date]['rain_mm'] += item.get('rain', {}).get('3h', 0)
            daily[date]['wind_speeds'].append(item['wind']['speed'])

        records = []
        for date, vals in daily.items():
            records.append({
                'city_id':   city_id,
                'city':      city_name,
                'date':      date,
                'temp_min':  round(min(vals['temps']), 1),
                'temp_max':  round(max(vals['temps']), 1),
                'humidity':  round(sum(vals['humidity']) / len(vals['humidity']), 1),
                'pop':       round(sum(vals['pop']) / len(vals['pop']), 2),
                'rain_mm':   round(vals['rain_mm'], 1),
                'wind_speed':round(sum(vals['wind_speeds']) / len(vals['wind_speeds']), 1),
            })
        return records
    except Exception as e:
        print(f'Erreur meteo {city_name}: {e}')
        return []

In [ ]:
all_weather = []
for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc='Meteo'):
    records = get_weather_forecast(row['lat'], row['lon'], row['city_id'], row['city'])
    all_weather.extend(records)
    time.sleep(0.3)

df_weather = pd.DataFrame(all_weather)
print(f'{len(df_weather)} enregistrements meteo')
print(f'Periode: {df_weather["date"].min()} -> {df_weather["date"].max()}')
df_weather.head(8)

---
# ETAPE 3 — Score Meteo & Classement

**Score composite pondere (0-100)** base sur 4 criteres :

| Critere | Poids | Logique |
|---------|-------|---------|
| Temperature max moyenne | 40% | Plus chaud = meilleur |
| Probabilite de pluie | 30% | Plus faible = meilleur |
| Volume de pluie total | 20% | Plus faible = meilleur |
| Humidite moyenne | 10% | Plus seche = meilleur |

In [ ]:
# Agregation par ville sur la periode
df_agg = df_weather.groupby(['city_id', 'city']).agg(
    temp_max_mean=('temp_max',  'mean'),
    pop_mean=     ('pop',       'mean'),
    rain_total=   ('rain_mm',   'sum'),
    humidity_mean=('humidity',  'mean'),
    wind_mean=    ('wind_speed','mean')
).reset_index()

def normalize(series, reverse=False):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    norm = (series - mn) / (mx - mn) * 100
    return (100 - norm) if reverse else norm

df_agg['score_temp'] = normalize(df_agg['temp_max_mean'])
df_agg['score_rain'] = normalize(df_agg['pop_mean'],     reverse=True)
df_agg['score_vol']  = normalize(df_agg['rain_total'],   reverse=True)
df_agg['score_hum']  = normalize(df_agg['humidity_mean'],reverse=True)

df_agg['weather_score'] = (
    df_agg['score_temp'] * 0.40 +
    df_agg['score_rain'] * 0.30 +
    df_agg['score_vol']  * 0.20 +
    df_agg['score_hum']  * 0.10
).round(2)

df_ranking = df_agg.sort_values('weather_score', ascending=False).reset_index(drop=True)
df_ranking['rank'] = df_ranking.index + 1

df_weather_final = df_ranking.merge(df_coords[['city_id', 'lat', 'lon']], on='city_id')

# Sauvegarde
df_weather_final.to_csv('weather_cities.csv', index=False)

print('Top 10 destinations :')
df_weather_final[['rank', 'city', 'temp_max_mean', 'pop_mean', 'rain_total', 'weather_score']].head(10)

---
# ETAPE 4 — Scraping Hotels sur Booking.com

**Selenium** (Chrome headless) + **BeautifulSoup** pour collecter 20 hotels par ville.

Donnees collectees : nom, URL, score, nb avis, description, coordonnees GPS.

> Note legale : scraping a usage academique uniquement.

In [ ]:
def create_driver():
    opts = Options()
    opts.add_argument('--headless')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def extract_score(card):
    tag = card.find('div', {'data-testid': 'review-score'})
    if tag:
        m = re.search(r'(\d+)[,.](\d+)', tag.get_text())
        if m:
            return float(f'{m.group(1)}.{m.group(2)}')
    return None

def scrape_city_hotels(driver, city_name, city_id, city_lat, city_lon, n_hotels=20):
    hotels = []
    city_enc = city_name.replace(' ', '+')
    driver.get(f'https://www.booking.com/searchresults.html?ss={city_enc}&lang=fr&dest_type=city')
    time.sleep(4)
    try:
        btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, 'onetrust-accept-btn-handler')))
        btn.click(); time.sleep(1)
    except Exception:
        pass
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="property-card"]')))
    except Exception:
        pass
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    cards = soup.find_all('div', {'data-testid': 'property-card'})
    for card in cards[:n_hotels]:
        name_tag = card.find('a', {'data-testid': 'title-link'})
        name = name_tag.get_text(strip=True) if name_tag else None
        if not name:
            continue
        url = name_tag['href'] if name_tag else ''
        if url and not url.startswith('http'):
            url = 'https://www.booking.com' + url
        url = url.split('?')[0] if '.html' in url else url
        score = extract_score(card)
        addr = card.find(['span','div'], {'data-testid': 'address'})
        hotels.append({
            'city_id': city_id, 'city': city_name,
            'hotel_name': name, 'url': url,
            'lat': city_lat, 'lon': city_lon,
            'score': score, 'description': addr.get_text(strip=True) if addr else ''
        })
    return hotels

In [ ]:
# Scraping (environ 10-15 min pour 35 villes)
np.random.seed(42)
driver = create_driver()
all_hotels = []

for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc='Hotels'):
    hotels = scrape_city_hotels(driver, row['city'], row['city_id'], row['lat'], row['lon'])
    all_hotels.extend(hotels)
    time.sleep(2)

driver.quit()

df_hotels = pd.DataFrame(all_hotels)
df_hotels['score'] = pd.to_numeric(df_hotels['score'], errors='coerce')

# Dispersion GPS legere
n = len(df_hotels)
df_hotels['lat'] += np.random.uniform(-0.02, 0.02, n)
df_hotels['lon'] += np.random.uniform(-0.02, 0.02, n)
df_hotels['hotel_id'] = range(1, n + 1)

df_hotels.to_csv('hotels_booking.csv', index=False)

print(f'{len(df_hotels)} hotels collectes')
print(f'Avec score: {df_hotels["score"].notna().sum()}')
df_hotels.dropna(subset=['score']).sort_values('score', ascending=False).head(10)[['hotel_name', 'city', 'score']]

---
# ETAPE 5 — Upload vers S3 (Data Lake)

**Data Lake** : stockage brut sur AWS S3.  
Architecture : `s3://alh-consulting-kayak/raw/*.csv`

In [ ]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

for filename in ['weather_cities.csv', 'hotels_booking.csv', 'weather_daily_detail.csv']:
    if os.path.exists(filename):
        s3_client.upload_file(filename, S3_BUCKET, f'raw/{filename}')
        print(f'OK  s3://{S3_BUCKET}/raw/{filename}')

# Verification
resp = s3_client.list_objects_v2(Bucket=S3_BUCKET, Prefix='raw/')
print(f'\nContenu bucket {S3_BUCKET}:')
for obj in resp.get('Contents', []):
    print(f"  {obj['Key']} ({obj['Size']/1024:.1f} KB)")

---
# ETAPE 6 — ETL vers AWS RDS (Data Warehouse)

**ETL = Extract, Transform, Load**

| | Data Lake (S3) | Data Warehouse (RDS) |
|--|----------------|---------------------|
| Donnees | Brutes | Nettoyees, structurees |
| Format | CSV | Tables SQL |
| Usage | Stockage, ML | Analyse, BI |

**Instance RDS** : `kayak-jedha.cdwsi2uiosiy.eu-west-3.rds.amazonaws.com`  
**Tables** : `cities_weather` + `hotels`

In [ ]:
# EXTRACT depuis S3
def read_s3_csv(key):
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

import io
df_weather_s3 = read_s3_csv('raw/weather_cities.csv')
df_hotels_s3  = read_s3_csv('raw/hotels_booking.csv')
print(f'Meteo S3: {df_weather_s3.shape}')
print(f'Hotels S3: {df_hotels_s3.shape}')

In [ ]:
# TRANSFORM
df_weather_clean = df_weather_s3[[
    'city_id', 'city', 'lat', 'lon',
    'temp_max_mean', 'pop_mean', 'rain_total',
    'humidity_mean', 'weather_score', 'rank'
]].copy()
df_weather_clean['temp_max_mean'] = df_weather_clean['temp_max_mean'].round(1)
df_weather_clean['weather_score'] = df_weather_clean['weather_score'].round(2)

df_hotels_clean = df_hotels_s3[[
    'city_id', 'city', 'hotel_name', 'url',
    'lat', 'lon', 'score', 'reviews', 'description'
]].copy()
df_hotels_clean['score'] = pd.to_numeric(df_hotels_clean['score'], errors='coerce')
df_hotels_clean['hotel_id'] = range(1, len(df_hotels_clean) + 1)

print(f'Meteo nettoyee: {df_weather_clean.shape}')
print(f'Hotels nettoyes: {df_hotels_clean.shape}')

In [ ]:
# LOAD vers RDS PostgreSQL
engine = create_engine(RDS_URI)

df_weather_clean.to_sql('cities_weather', engine, if_exists='replace', index=False)
print(f"Table 'cities_weather' chargee: {len(df_weather_clean)} lignes")

df_hotels_clean.to_sql('hotels', engine, if_exists='replace', index=False)
print(f"Table 'hotels' chargee: {len(df_hotels_clean)} lignes")

In [ ]:
# Verification depuis RDS
with engine.connect() as conn:
    top5 = pd.read_sql(
        text('SELECT city, weather_score, rank FROM cities_weather ORDER BY rank LIMIT 5'),
        conn
    )
    print('Top 5 destinations (depuis RDS):')
    print(top5.to_string(index=False))

    top_hotels = pd.read_sql(
        text('SELECT hotel_name, city, score FROM hotels WHERE score IS NOT NULL ORDER BY score DESC LIMIT 5'),
        conn
    )
    print('\nTop 5 hotels (depuis RDS):')
    print(top_hotels.to_string(index=False))

---
# ETAPE 7 — Visualisation avec Plotly

Deux cartes interactives :
1. **Top-5 destinations** (score meteo sur 5 jours)
2. **Top-20 hotels** dans les meilleures villes

In [ ]:
# CARTE 1 : Toutes les destinations avec score meteo
df_map = df_weather_final.copy()

fig_destinations = px.scatter_mapbox(
    df_map,
    lat='lat', lon='lon',
    color='weather_score',
    size='weather_score',
    hover_name='city',
    hover_data={
        'weather_score': ':.1f',
        'temp_max_mean': ':.1f',
        'pop_mean': ':.1%',
        'rain_total': ':.1f',
        'rank': True
    },
    color_continuous_scale='RdYlGn',
    size_max=30,
    zoom=4.5,
    center={'lat': 46.5, 'lon': 2.5},
    mapbox_style='open-street-map',
    title='<b>Meilleures Destinations en France</b><br><sub>Score meteo sur 5 jours</sub>',
    labels={
        'weather_score': 'Score meteo',
        'temp_max_mean': 'Temp max moy (C)',
        'pop_mean': 'Proba pluie',
        'rain_total': 'Pluie totale (mm)',
        'rank': 'Classement'
    }
)

# Etoiles pour le Top 5
top5 = df_map[df_map['rank'] <= 5]
fig_destinations.add_trace(go.Scattermapbox(
    lat=top5['lat'], lon=top5['lon'],
    mode='markers+text',
    text=top5['rank'].astype(str) + '. ' + top5['city'],
    textposition='top right',
    marker=dict(size=20, color='gold', symbol='star'),
    name='Top 5', hoverinfo='skip'
))

fig_destinations.update_layout(height=600, margin={'r':0,'t':70,'l':0,'b':0})
fig_destinations.show()

In [ ]:
# CARTE 2 : Top-20 hotels dans le Top 5 des destinations
top5_cities = df_weather_final[df_weather_final['rank'] <= 5]['city'].tolist()
df_hotels_top = (
    df_hotels_clean[df_hotels_clean['city'].isin(top5_cities)]
    .dropna(subset=['score'])
    .sort_values('score', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
df_hotels_top['hotel_rank'] = df_hotels_top.index + 1

fig_hotels = px.scatter_mapbox(
    df_hotels_top,
    lat='lat', lon='lon',
    color='score',
    size='score',
    hover_name='hotel_name',
    hover_data={'city': True, 'score': ':.1f', 'description': True},
    color_continuous_scale='Blues',
    size_max=20,
    zoom=4.5,
    center={'lat': 44.5, 'lon': 5.0},
    mapbox_style='open-street-map',
    title='<b>Top 20 Hotels dans les Meilleures Destinations</b><br><sub>Aix-en-Provence, Avignon, Marseille, Paris, Grenoble</sub>',
    labels={'score': 'Score Booking', 'city': 'Ville'}
)
fig_hotels.update_layout(height=600, margin={'r':0,'t':70,'l':0,'b':0})
fig_hotels.show()

print('Top 20 hotels :')
df_hotels_top[['hotel_rank', 'hotel_name', 'city', 'score']].to_string(index=False)

---
# CONCLUSION

## Resultats obtenus

| Etape | Resultat | Technologie |
|-------|---------|-------------|
| GPS | 35 villes geolocalises | Nominatim API |
| Meteo | 175 previsions (5j x 35 villes) | OpenWeatherMap API |
| Score | Classement pondere | Pandas |
| Hotels | 679 hotels scrapes | Selenium + BS4 |
| Data Lake | 3 fichiers CSV | AWS S3 (alh-consulting-kayak) |
| Data Warehouse | 2 tables SQL | AWS RDS PostgreSQL (eu-west-3) |
| Visualisation | 2 cartes interactives | Plotly |

## Top 5 Destinations (score meteo)
1. **Aix-en-Provence** — Score 88.91
2. **Avignon** — Score 87.81
3. **Marseille** — Score 86.65
4. **Paris** — Score 85.54
5. **Grenoble** — Score 82.25

## Infrastructure AWS
- **S3** : `s3://alh-consulting-kayak/raw/`
- **RDS** : `kayak-jedha.cdwsi2uiosiy.eu-west-3.rds.amazonaws.com`
- **Region** : `eu-west-3` (Paris)